# Sticky ZigZag: sampler behavior on a CNN posterior

Loads a saved run from `fast_mnist_cnn.py` and asks the questions that
actually matter for using a sticky PDMP sampler on a network this size
(with only `n_events` skeleton points, not all `D` coordinates can be
expected to move): does the bound hold, do parameters move at all, how do
the samples compare to the MAP reference, does the sticky measure keep the
*right* coordinates at exact zero (e.g. MNIST's all-black border pixels),
are there regions of parameter space that never move, and -- ultimately --
are the resulting predictions accurate and usefully uncertain.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
%matplotlib inline

if Path.cwd().name == "notebooks":
    os.chdir("..")

plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "font.size":          11,
})

## 1. Load runs

In [ ]:
RUN_DIR = Path("results/grid/mnist_cnn/split_00")

RUN_SPECS = [
    ("zigzag_n10000", "grid_sticky_zigzag.pt", 10_000),
    ("boomerang_n10000", "grid_sticky_boomerang.pt", 10_000),
]

runs = {}
for label, fname, n_train in RUN_SPECS:
    path = RUN_DIR / fname
    if not path.exists():
        print(f"[{label}] skipped -- {path} not found")
        continue
    ckpt = torch.load(path, weights_only=False)
    runs[label] = {"ckpt": ckpt, "n_train": n_train, "D": ckpt["x_ref"].shape[0]}
    has_diag = ckpt.get("diagnostics") is not None
    print(f"[{label}] {fname}  sampler={ckpt['sampler']}  D={ckpt['x_ref'].shape[0]}  "
          f"bound_violations={ckpt['bound_violations']}  diagnostics saved={has_diag}")

assert runs, f"No run files found under {RUN_DIR} -- check RUN_SPECS against what's actually on disk."

## 2. Sparsity: cold start vs. resampled draws

`prune_frac` is the fraction frozen at t=0; `sparsity_frac` (saved in the
checkpoint) is the fraction of near-zero coordinates across the resampled
draws. Comparing them shows whether the sampler thaws over the run
(sparsity should drop) or holds/increases (freezing more than it thaws).

In [ ]:
sparsity_rows = []
for label, r in runs.items():
    ckpt = r["ckpt"]
    samples = ckpt["samples"]
    per_draw_sparsity = (samples.abs() < 1e-8).float().mean(dim=1)
    sparsity_rows.append({
        "run": label,
        "prune_frac (cold start)": ckpt["prune_frac"],
        "sparsity_frac (saved, resampled draws)": ckpt["sparsity_frac"],
        "per_draw sparsity min": float(per_draw_sparsity.min()),
        "per_draw sparsity max": float(per_draw_sparsity.max()),
        "bound_violations": ckpt["bound_violations"],
        "elapsed_sec": ckpt["elapsed_sec"],
    })

sparsity_df = pd.DataFrame(sparsity_rows).set_index("run")
display(sparsity_df.style.format({
    "prune_frac (cold start)": "{:.4f}", "sparsity_frac (saved, resampled draws)": "{:.4f}",
    "per_draw sparsity min": "{:.4f}", "per_draw sparsity max": "{:.4f}", "elapsed_sec": "{:.1f}",
}))

fig, ax = plt.subplots(figsize=(8, 3.5))
for label, r in runs.items():
    per_draw = (r["ckpt"]["samples"].abs() < 1e-8).float().mean(dim=1)
    ax.plot(per_draw.cpu(), marker="o", markersize=3, label=label)
ax.set_xlabel("resampled draw index")
ax.set_ylabel("fraction of D near zero")
ax.set_title("Sparsity across the resampled path")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 3. Per-iteration diagnostics + validity check

Builds one DataFrame per run from `ckpt["diagnostics"]` (only present for
non-chunked ZigZag runs). Everything below is derived from this.

In [ ]:
diag_dfs = {}
for label, r in runs.items():
    diag = r["ckpt"].get("diagnostics")
    if diag is None:
        print(f"[{label}] no per-iteration diagnostics saved (chunked run, or predates this field) -- skipped below")
        continue
    df = pd.DataFrame(diag)
    df["iteration"] = np.arange(len(df))
    df["violated"] = df["bound_violations"] > 0
    diag_dfs[label] = df
    print(f"[{label}] {len(df)} iterations  |  {int(df['violated'].sum())} violating iterations "
          f"({100 * df['violated'].mean():.2f}%)")

In [ ]:
for label, df in diag_dfs.items():
    n = len(df)
    n_viol = int(df["violated"].sum())
    if n_viol == 0:
        print(f"[{label}] {n} iterations, 0 bound violations -- grid bound held everywhere.")
    else:
        viol = df[df["violated"]]
        print(f"[{label}] {n_viol}/{n} iterations violated the bound "
              f"({100 * n_viol / n:.2f}%) -- max_ratio at violations: "
              f"median={viol['max_ratio'].median():.3f}  max={viol['max_ratio'].max():.3f}")

## 4. Event mix and adaptive horizon

Coarse behavior of the sticky loop over the run: how the event mix
(freeze/thaw/bounce) settles, and how the adaptive grid horizon `t_max`
moves.

In [ ]:
fig, axes = plt.subplots(len(runs), 1, figsize=(9, 2.8 * len(runs)), squeeze=False)
for ax_row, (label, r) in zip(axes, runs.items()):
    ax = ax_row[0]
    tmax_log = r["ckpt"].get("grid_t_max_log")
    if not tmax_log:
        ax.set_title(f"{label}: no grid_t_max_log saved")
        continue
    ax.plot(tmax_log, lw=0.8)
    ax.set_yscale("log")
    ax.set_xlabel("_grid_bound call index")
    ax.set_ylabel("t_max (log scale)")
    ax.set_title(f"{label}: adaptive horizon over the run")
fig.tight_layout()
plt.show()

In [ ]:
for label, df in diag_dfs.items():
    binding_counts = df["binding"].value_counts()
    event_counts = df["event_type"].value_counts()
    print(f"[{label}] binding term counts:\n{binding_counts}\n")
    print(f"[{label}] event_type counts:\n{event_counts}\n")

## 5. Do parameters actually move?

`n_events=5000` skeleton points on a `D~60k` model means most coordinates
cannot be expected to move even once -- movement happens through two
different mechanisms and they're worth separating:

- **bounces** (`flipped_coord`): a coordinate's *velocity* flips, i.e. the
  sampler actually reversed direction on it because of a genuine
  rate-function event. This is the "the posterior curvature pushed back
  here" signal.
- **thaw events**: a frozen coordinate becomes active (leaves exact zero)
  and starts moving deterministically at constant velocity until its next
  freeze/bounce. A coordinate can thaw and re-freeze without ever bouncing.

So "did this coordinate ever move" really has three states: **never left
cold-start freeze**, **thawed but never bounced** (moved, but only ever
under unopposed drift -- weak/no local signal), and **bounced at least
once** (moved under a real curvature signal). This cell classifies every
coordinate into those three buckets.

In [ ]:
movement_rows = []
movement_masks = {}  # label -> dict of D-length bool masks, reused in later sections
for label, r in runs.items():
    ckpt = r["ckpt"]
    D = r["D"]
    cold_frozen = ckpt["cold_start_mask"].cpu()
    samples = ckpt["samples"].cpu()

    # "ever nonzero" is the direct readout of "left exact zero at some point"
    # -- covers thaw-without-bounce movement, which flipped_coord can't see.
    ever_nonzero = (samples.abs() > 1e-8).any(dim=0)

    df = diag_dfs.get(label)
    if df is not None and "flipped_coord" in df.columns:
        flips = df["flipped_coord"].dropna().astype(int)
        bounced = torch.zeros(D, dtype=torch.bool)
        if len(flips):
            bounced[torch.as_tensor(flips.to_numpy().copy())] = True
        n_bounce_events = int(len(flips))
    else:
        bounced = torch.zeros(D, dtype=torch.bool)
        n_bounce_events = None

    never_left_zero = cold_frozen & ~ever_nonzero
    thawed_never_bounced = ever_nonzero & ~bounced
    bounced_at_least_once = bounced

    movement_masks[label] = {
        "cold_frozen": cold_frozen, "ever_nonzero": ever_nonzero,
        "bounced": bounced, "never_left_zero": never_left_zero,
        "thawed_never_bounced": thawed_never_bounced,
    }
    movement_rows.append({
        "run": label,
        "D": D,
        "cold-start frozen": int(cold_frozen.sum()),
        "never left zero": int(never_left_zero.sum()),
        "thawed, never bounced": int(thawed_never_bounced.sum()),
        "bounced >=1x": int(bounced_at_least_once.sum()),
        "distinct coords bounced": int(bounced.sum()),
        "total bounce events": n_bounce_events,
    })

movement_df = pd.DataFrame(movement_rows).set_index("run")
movement_df["frac never moved"] = movement_df["never left zero"] / movement_df["D"]
movement_df["frac bounced"] = movement_df["bounced >=1x"] / movement_df["D"]
display(movement_df.style.format({"frac never moved": "{:.4f}", "frac bounced": "{:.4f}"}))

print(
    "Read: 'bounced >=1x' is the set of coordinates for which the sampler has "
    "actual evidence of local curvature -- everything else (including "
    "'thawed, never bounced') has only ever moved under unopposed ballistic "
    "drift between events, or not moved at all. With n_events=5000 on D~60k, "
    "a small 'bounced' fraction is expected and not itself a problem; what "
    "matters is WHICH coordinates end up in which bucket (sections 6-7)."
)

## 6. Layer map

To ask "which regions of the model move" or "which weights read the
MNIST border" we need to unflatten a coordinate index back to
`(layer_name, position within tensor)`. This replicates
`BayesianModule.build`'s own flattening (`model.py`: `names`/`shapes` from
`module.named_parameters()`, walked in order) directly from the
checkpoint's `activation`/`D`, without paying to reconstruct the full
`BayesianModule` (data loading + target build) -- that only happens in
section 9 where we actually need a forward pass.

Architecture is `LeNet5` if `D==61706`, `CNN` if `D==19466` (the two
options wired up in `fast_mnist_cnn.py`'s `ARCHITECTURES`); this asserts
rather than guessing silently if a run doesn't match either.

In [ ]:
# (name, shape) in module.named_parameters() order -- must match neural_networks.py exactly.
LAYER_SHAPES = {
    "cnn": [
        ("conv1.weight", (32, 1, 3, 3)), ("conv1.bias", (32,)),
        ("conv2.weight", (64, 32, 3, 3)), ("conv2.bias", (64,)),
        ("fc.weight", (10, 64)), ("fc.bias", (10,)),
    ],
    "lenet5": [
        ("conv1.weight", (6, 1, 5, 5)), ("conv1.bias", (6,)),
        ("conv2.weight", (16, 6, 5, 5)), ("conv2.bias", (16,)),
        ("fc1.weight", (120, 400)), ("fc1.bias", (120,)),
        ("fc2.weight", (84, 120)), ("fc2.bias", (84,)),
        ("fc3.weight", (10, 84)), ("fc3.bias", (10,)),
    ],
}
D_TO_ARCH = {sum(int(np.prod(s)) for _, s in layers): arch for arch, layers in LAYER_SHAPES.items()}


def build_layer_index(D: int) -> pd.DataFrame:
    """Per-coordinate DataFrame: layer name, shape, within-tensor position, is_bias."""
    if D not in D_TO_ARCH:
        raise ValueError(f"D={D} doesn't match a known architecture ({D_TO_ARCH}) -- "
                          f"add its (name, shape) list to LAYER_SHAPES above.")
    layers = LAYER_SHAPES[D_TO_ARCH[D]]
    rows = []
    idx = 0
    for name, shape in layers:
        n = int(np.prod(shape))
        is_bias = len(shape) == 1
        for local_i in range(n):
            pos = np.unravel_index(local_i, shape) if not is_bias else (local_i,)
            rows.append({"coord": idx, "layer": name, "is_bias": is_bias, "shape": shape, "pos": pos})
            idx += 1
    return pd.DataFrame(rows).set_index("coord")


layer_idx_dfs = {label: build_layer_index(r["D"]) for label, r in runs.items()}
for label, ldf in layer_idx_dfs.items():
    print(f"[{label}] architecture={D_TO_ARCH[runs[label]['D']]}  layers={ldf['layer'].unique().tolist()}")

## 7. Where in the network does movement happen? Stuck regions

Per-layer breakdown of the three movement buckets from section 5. A layer
that's almost entirely "never left zero" or "thawed, never bounced" is a
candidate "stuck region" -- either genuinely flat posterior curvature
there, or a sign `n_events` is too small / the sampler isn't mixing that
part of the model. Biases are shown separately since `can_freeze` is
always `False` for them (they can never cold-start-freeze, so any bias in
"never left zero" got frozen dynamically during the run, not at t=0).

In [ ]:
for label, masks in movement_masks.items():
    ldf = layer_idx_dfs[label].copy()
    ldf["never_left_zero"] = masks["never_left_zero"].numpy()
    ldf["thawed_never_bounced"] = masks["thawed_never_bounced"].numpy()
    ldf["bounced"] = masks["bounced"].numpy()

    by_layer = ldf.groupby("layer").agg(
        n=("layer", "size"),
        never_left_zero=("never_left_zero", "sum"),
        thawed_never_bounced=("thawed_never_bounced", "sum"),
        bounced=("bounced", "sum"),
    )
    by_layer["frac_never_moved"] = by_layer["never_left_zero"] / by_layer["n"]
    by_layer["frac_bounced"] = by_layer["bounced"] / by_layer["n"]
    # preserve network order (groupby sorts alphabetically otherwise)
    order = [name for name, _ in LAYER_SHAPES[D_TO_ARCH[runs[label]["D"]]]]
    by_layer = by_layer.loc[order]

    print(f"[{label}]")
    display(by_layer.style.format({"frac_never_moved": "{:.4f}", "frac_bounced": "{:.4f}"})
            .background_gradient(subset=["frac_never_moved"], cmap="Reds", vmin=0, vmax=1)
            .background_gradient(subset=["frac_bounced"], cmap="Greens", vmin=0, vmax=1))

    fig, ax = plt.subplots(figsize=(9, 3.5))
    x = np.arange(len(by_layer))
    width = 0.25
    ax.bar(x - width, by_layer["never_left_zero"] / by_layer["n"], width, label="never left zero", color="#c44e52")
    ax.bar(x, by_layer["thawed_never_bounced"] / by_layer["n"], width, label="thawed, never bounced", color="#dd8452")
    ax.bar(x + width, by_layer["bounced"] / by_layer["n"], width, label="bounced >=1x", color="#55a868")
    ax.set_xticks(x)
    ax.set_xticklabels(by_layer.index, rotation=30, ha="right")
    ax.set_ylabel("fraction of layer's coordinates")
    ax.set_title(f"{label}: movement bucket by layer")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## 8. How different are the samples from the reference (MAP) point?

`x_ref` is the checkpoint's own cold-start / MAP reference. Per-layer
`|sample - x_ref|` (relative to that coordinate's prior std, so layers
with very different weight scales -- e.g. `conv1` vs `fc3` -- are
comparable) tells us whether the posterior is actually exploring around
`x_ref` or just sitting on it. Combined with section 7: a layer can show
"bounced >=1x" for many coordinates but still have tiny displacement if
those bounces only ever nudge it a little.

In [ ]:
# Optional: a MAP-reference checkpoint (cnn_reference.py / lenet_reference.py output) that
# carries Sigma_inv = prior_precision + fisher_diag, for scale-normalizing displacement.
# Matched to grid_sticky_zigzag.pt (D=61706, LeNet5) -- point this elsewhere if RUN_SPECS changes.
MAP_REF_PATHS = {
    "lenet5": Path("results/maps/lenet_reference_N60000_steps10000.pt"),
    "cnn": Path("results/maps/cnn_reference_N60000_steps10000.pt"),
}

sigma_inv_by_D = {}
for arch, path in MAP_REF_PATHS.items():
    if path.exists():
        map_ckpt = torch.load(path, map_location="cpu", weights_only=False)
        sigma_inv_by_D[map_ckpt["x_ref"].shape[0]] = map_ckpt["Sigma_inv"].cpu()
        print(f"[{arch}] loaded Sigma_inv from {path}")
    else:
        print(f"[{arch}] {path} not found -- displacement will be shown in raw units for matching D")

displacement_rows = []
displacement_by_coord = {}
for label, r in runs.items():
    ckpt = r["ckpt"]
    D = r["D"]
    x_ref = ckpt["x_ref"].cpu()
    samples = ckpt["samples"].cpu()
    abs_disp = (samples - x_ref).abs()  # [n_events, D]

    sigma_inv = sigma_inv_by_D.get(D)
    if sigma_inv is not None:
        prior_std = sigma_inv.clamp(min=1e-12).rsqrt()
        scaled_disp = abs_disp / prior_std
        unit = "prior std"
    else:
        scaled_disp = abs_disp
        unit = "raw"

    mean_disp = scaled_disp.mean(dim=0)  # [D], averaged over draws
    displacement_by_coord[label] = mean_disp
    ldf = layer_idx_dfs[label].copy()
    ldf["mean_disp"] = mean_disp.numpy()
    by_layer = ldf.groupby("layer")["mean_disp"].agg(["mean", "median", "max"])
    order = [name for name, _ in LAYER_SHAPES[D_TO_ARCH[D]]]
    by_layer = by_layer.loc[order]
    by_layer.columns = pd.MultiIndex.from_product([[f"{label} ({unit})"], by_layer.columns])
    displacement_rows.append(by_layer)

displacement_df = pd.concat(displacement_rows, axis=1)
display(displacement_df.style.format("{:.4f}").background_gradient(cmap="viridis", axis=None))

## 9. Sparsity fidelity: does the sticky measure zero out the MNIST border?

MNIST digits are always centered with an all-black margin -- the outer few
pixels carry essentially no information, and a continuous-parameter BNN
can't represent "this weight is truly irrelevant" exactly, only "this
weight has a very concentrated posterior near zero." A sticky sampler's
atomic point mass at zero *can* represent that faithfully. This checks
whether it actually does, for `conv1` (the only layer that reads raw
pixels directly).

`conv1` is `Conv2d(1, 6, kernel_size=5)` applied to the 28x28 image
zero-padded to 32x32 (`LeNet5.forward`: `F.pad(x, [2,2,2,2])`, then
`conv1`). A kernel weight at offset `(kh, kw)` for output pixel `(oh, ow)`
reads padded-image pixel `(oh+kh, ow+kw)`; sliding over all 28x28 output
positions, every offset touches *some* real pixels eventually (the 2px
synthetic pad is thin relative to the 5x5 kernel), so there's no exact
"never touches a real pixel" weight. Instead we use the actual training
images: MNIST's own margin is near-constant black over a much wider ring
than just the 2px synthetic pad (its per-pixel variance across the train
set is ~0 well into the interior). For each `conv1` weight we compute its
usage-weighted average per-pixel *variance* -- the variance, over the
training set, of the pixels that weight actually reads from, averaged
over its 28x28 uses. A weight with near-zero average variance only ever
sees pixels that carry no information regardless of the digit -- exactly
the case where an atomic prior mass at zero is the *correct* faithful
representation, not an approximation.

In [ ]:
from torchvision import datasets, transforms

# Per-pixel variance of the padded MNIST image, over a representative sample of the
# actual training set (matches load_mnist_subset's own transform: ToTensor + Normalize).
_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
_mnist_train = datasets.MNIST("datasets", train=True, download=True, transform=_transform)
_rng = np.random.default_rng(0)
_sample_idx = _rng.choice(len(_mnist_train), size=2000, replace=False)
_imgs = torch.stack([_mnist_train[int(i)][0] for i in _sample_idx]).squeeze(1)  # [N, 28, 28]
_imgs_padded = torch.nn.functional.pad(_imgs, [2, 2, 2, 2])  # [N, 32, 32]
pixel_var = _imgs_padded.var(dim=0)  # [32, 32], var=0 in the synthetic pad by construction

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(pixel_var, cmap="viridis")
ax.set_title("Per-pixel variance across 2000 training images\n(padded to 32x32)")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()

# # Usage-weighted mean pixel-variance seen by each conv1 weight offset (kh, kw).
# KERNEL, OUT = 5, 28
# offset_mean_var = torch.zeros(KERNEL, KERNEL)
# for kh in range(KERNEL):
#     for kw in range(KERNEL):
#         patch = pixel_var[kh:kh + OUT, kw:kw + OUT]  # all (oh, ow) uses of this offset
#         offset_mean_var[kh, kw] = patch.mean()

# lo, hi = offset_mean_var.min().item(), offset_mean_var.max().item()
# fig, ax = plt.subplots(figsize=(4.3, 4))
# im = ax.imshow(offset_mean_var, cmap="viridis", vmin=lo, vmax=hi)
# ax.set_title("conv1 weight offset -> mean pixel variance it reads")
# for (kh, kw), val in np.ndenumerate(offset_mean_var.numpy()):
#     ax.text(kw, kh, f"{val:.4f}", ha="center", va="center", color="white", fontsize=7)
# cbar = fig.colorbar(im, ax=ax, fraction=0.046)
# cbar.formatter.set_useOffset(False)
# cbar.update_ticks()
# fig.tight_layout()
# plt.show()

# rel_spread = (hi - lo) / offset_mean_var.mean().item()
# print(
#     f"offset_mean_var ranges {lo:.4f} to {hi:.4f} ({100*rel_spread:.1f}% relative spread). "
#     f"A 5x5 kernel sliding over a 28x28 output on a 32x32 canvas has EVERY offset visit "
#     f"nearly the same mix of pixels (each offset's receptive field differs from its "
#     f"neighbor's by only a 1-pixel shift out of 28) -- so at LeNet5's conv1 scale, kernel "
#     f"OFFSET alone barely distinguishes border from center. Any zeroing of true "
#     f"border-only information has to come from output CHANNEL/spatial structure learned "
#     f"jointly across the 6 filters, not from a single weight's fixed geometric position -- "
#     f"the next cell checks whether movement correlates with this (weak, as expected) signal "
#     f"anyway, and section 7's per-layer view is the more informative read on where sparsity "
#     f"actually concentrates."
# )

In [ ]:
from torchvision import datasets, transforms

# Per-pixel variance of the padded MNIST image, over a representative sample of the
# actual training set (matches load_mnist_subset's own transform: ToTensor + Normalize).
_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
_mnist_train = datasets.MNIST("datasets", train=True, download=True, transform=_transform)
_rng = np.random.default_rng(0)
_sample_idx = _rng.choice(len(_mnist_train), size=2000, replace=False)
_imgs = torch.stack([_mnist_train[int(i)][0] for i in _sample_idx]).squeeze(1)  # [N, 28, 28]
_imgs_padded = torch.nn.functional.pad(_imgs, [2, 2, 2, 2])  # [N, 32, 32]
pixel_var = _imgs_padded.var(dim=0)  # [32, 32], var=0 in the synthetic pad by construction

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(pixel_var, cmap="viridis")
ax.set_title("Per-pixel variance across 2000 training images\n(padded to 32x32)")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()

# Usage-weighted mean pixel-variance seen by each conv1 weight offset (kh, kw)
# -- kept for section 8's correlation against posterior movement.
KERNEL, OUT = 5, 28
offset_mean_var = torch.zeros(KERNEL, KERNEL)
for kh in range(KERNEL):
    for kw in range(KERNEL):
        patch = pixel_var[kh:kh + OUT, kw:kw + OUT]  # all (oh, ow) uses of this offset
        offset_mean_var[kh, kw] = patch.mean()

# Now the analogous question for the SAMPLED models: instead of "how much does this
# pixel vary across training images", ask "how much does conv1's OUTPUT on a real digit
# vary across posterior draws" -- i.e. push actual posterior samples through conv1 on one
# real MNIST image and look at the resulting activation maps, the same imshow style as
# above but downstream of the sampler instead of the raw data. Both runs share one figure
# so they're directly comparable side by side.
RUN_DISPLAY_NAME = {
    "zigzag_n10000": "Sticky Zig-Zag",
    "boomerang_n10000": "Sticky Boomerang",
}

N_DRAWS_SHOWN = 1000
_digit_idx = int(_sample_idx[5])
_digit_img = _mnist_train[_digit_idx][0].squeeze(0)  # [28, 28]
_digit_padded = torch.nn.functional.pad(_digit_img, [2, 2, 2, 2]).unsqueeze(0).unsqueeze(0)  # [1, 1, 32, 32]

lenet_runs = [(label, r) for label, r in runs.items() if D_TO_ARCH.get(r["D"]) == "lenet5"]
skipped = [label for label, r in runs.items() if D_TO_ARCH.get(r["D"]) != "lenet5"]
for label in skipped:
    print(f"[{label}] not LeNet5 -- conv1 forward pass above is LeNet5-specific, skip")

n_runs = len(lenet_runs)
fig, axes = plt.subplots(2 * n_runs, 7, figsize=(14, 4.2 * n_runs))
if n_runs == 1:
    axes = axes.reshape(2, 7)

for row_block, (label, r) in enumerate(lenet_runs):
    ldf = layer_idx_dfs[label]
    w_coords = torch.as_tensor(ldf[ldf["layer"] == "conv1.weight"].index.to_numpy())
    b_coords = torch.as_tensor(ldf[ldf["layer"] == "conv1.bias"].index.to_numpy())

    samples = r["ckpt"]["samples"].cpu()
    n_draws = min(N_DRAWS_SHOWN, samples.shape[0])
    draw_idx = torch.randperm(samples.shape[0])[:n_draws]

    acts = []
    with torch.no_grad():
        for i in draw_idx:
            w = samples[i, w_coords].view(6, 1, 5, 5).float()
            b = samples[i, b_coords].float()
            acts.append(torch.nn.functional.conv2d(_digit_padded.float(), w, b).squeeze(0))  # [6, 28, 28]
    acts = torch.stack(acts)  # [n_draws, 6, 28, 28]

    mean_act = acts.mean(dim=0)
    std_act = acts.std(dim=0)

    mean_row, std_row = axes[2 * row_block], axes[2 * row_block + 1]
    display_name = RUN_DISPLAY_NAME.get(label, label)

    mean_row[0].imshow(_digit_img, cmap="gray")
    mean_row[0].set_title("input digit", fontsize=9)
    std_row[0].axis("off")
    std_row[0].text(0.5, 0.5, display_name, fontsize=11, fontweight="semibold",
                     ha="center", va="center", transform=std_row[0].transAxes)
    for c in range(6):
        mean_row[c + 1].imshow(mean_act[c], cmap="viridis")
        mean_row[c + 1].set_title(f"ch{c} mean act.", fontsize=9)
        std_row[c + 1].imshow(std_act[c], cmap="viridis")
        std_row[c + 1].set_title(f"ch{c} std across draws", fontsize=9)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Posterior uncertainty in conv1 concentrates on the digit's strokes, not the background", fontsize=12)
fig.tight_layout()
plt.show()


In [ ]:
# Now correlate offset_mean_var against whether the sampler actually keeps that weight
# at (or very near) zero -- both at cold start and across the resampled posterior.
for label, r in runs.items():
    D = r["D"]
    if D_TO_ARCH.get(D) != "lenet5":
        print(f"[{label}] not LeNet5 (D={D}) -- conv1 geometry above is LeNet5-specific, skip")
        continue
    ckpt = r["ckpt"]
    ldf = layer_idx_dfs[label]
    conv1_w = ldf[ldf["layer"] == "conv1.weight"].copy()
    # pos is (out_channel, in_channel, kh, kw); in_channel is always 0 here (in_channels=1)
    conv1_w["kh"] = conv1_w["pos"].apply(lambda p: p[2])
    conv1_w["kw"] = conv1_w["pos"].apply(lambda p: p[3])
    conv1_w["pixel_var"] = conv1_w.apply(lambda row: offset_mean_var[row["kh"], row["kw"]].item(), axis=1)

    coords = conv1_w.index.to_numpy()
    conv1_w["cold_frozen"] = movement_masks[label]["cold_frozen"][coords].numpy()
    conv1_w["never_left_zero"] = movement_masks[label]["never_left_zero"][coords].numpy()
    conv1_w["mean_abs_disp"] = displacement_by_coord[label][coords].numpy()

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].scatter(conv1_w["pixel_var"], conv1_w["mean_abs_disp"], s=10, alpha=0.5)
    axes[0].set_xlabel("mean pixel variance this weight reads")
    axes[0].set_ylabel("mean |sample - x_ref| (displacement)")
    axes[0].set_title(f"{label}: conv1 weights -- pixel informativeness vs. movement")

    by_offset = conv1_w.groupby(["kh", "kw"]).agg(
        pixel_var=("pixel_var", "first"),
        frac_cold_frozen=("cold_frozen", "mean"),
        frac_never_left_zero=("never_left_zero", "mean"),
        mean_disp=("mean_abs_disp", "mean"),
    ).reset_index()
    axes[1].scatter(by_offset["pixel_var"], by_offset["frac_never_left_zero"], s=40, label="frac never left zero")
    axes[1].scatter(by_offset["pixel_var"], by_offset["frac_cold_frozen"], s=20, marker="x", label="frac cold-frozen")
    axes[1].set_xlabel("mean pixel variance (per kernel offset)")
    axes[1].set_ylabel("fraction of that offset's weights")
    axes[1].set_title(f"{label}: does low-info offset -> stays frozen?")
    axes[1].legend(fontsize=8)
    axes[1].set_ylim(-0.05, 1.05)
    fig.tight_layout()
    plt.show()

    corr = conv1_w[["pixel_var", "mean_abs_disp"]].corr().iloc[0, 1]
    print(f"[{label}] corr(pixel_var, mean displacement) across conv1 weights = {corr:.3f}  "
          f"(positive = weights reading more-informative pixels move more, as expected "
          f"if the sticky measure is faithfully tracking pixel informativeness)")

## 10. Performance: accuracy and predictive uncertainty

`test_accuracy` in the checkpoint is already posterior-averaged (softmax
probabilities averaged over a 300-draw subsample, see
`evaluate_accuracy` in `fast_mnist_cnn.py`) but is a single number with no
uncertainty breakdown. This section rebuilds the target (loads MNIST +
the network, matching `build_target`) to get per-test-point predictive
probabilities, then reports accuracy, log-likelihood, and predictive
entropy (`sazz/utils/metrics.py: classification_metrics`) -- and compares
posterior-averaged probabilities against the single MAP point, since the
gap between them *is* the value the sampler is adding over a point
estimate. This cell does real CNN forward passes (subsampled draws, not
all `n_events`) so it's the most expensive cell in the notebook.

In [ ]:
from sazz.gpu_friendly.models.neural_networks import CNN, LeNet5
from sazz.gpu_friendly.models.model import to_param_dict
from sazz.gpu_friendly.scripts.fast_mnist_cnn import load_mnist_subset, N_TRAIN, N_TEST, BASE_SEED, DATA_DIR
from sazz.utils.metrics import classification_metrics

ARCHITECTURES = {"cnn": CNN, "lenet5": LeNet5}
N_UNCERTAINTY_DRAWS = 300  # matches evaluate_accuracy's own N_ACCURACY_DRAWS

print("Loading MNIST test set (same seed/split as the run) ...")
data = load_mnist_subset(N_TRAIN, N_TEST, BASE_SEED, DATA_DIR, dtype=torch.float64, device="cpu")
X_test, y_test = data["X_test"], data["y_test"]

perf_rows = []
predictive_probs = {}
for label, r in runs.items():
    ckpt = r["ckpt"]
    D = r["D"]
    arch = D_TO_ARCH.get(D)
    if arch is None:
        print(f"[{label}] unrecognized D={D} -- skip")
        continue

    module = ARCHITECTURES[arch](activation=ckpt["activation"], pool=ckpt["pool"]).to(dtype=torch.float64)
    names = [n for n, _ in module.named_parameters()]
    shapes = [p.shape for _, p in module.named_parameters()]
    param_dict_fn = to_param_dict(names, shapes)

    x_ref = ckpt["x_ref"].to(dtype=torch.float64)
    samples = ckpt["samples"].to(dtype=torch.float64)
    idx = torch.randperm(samples.shape[0])[:min(N_UNCERTAINTY_DRAWS, samples.shape[0])]
    sub = samples[idx]

    with torch.no_grad():
        map_logits = torch.func.functional_call(module, param_dict_fn(x_ref), (X_test,))
        map_probs = torch.softmax(map_logits, dim=-1)

        draw_probs = []
        for beta in sub:
            logits = torch.func.functional_call(module, param_dict_fn(beta), (X_test,))
            draw_probs.append(torch.softmax(logits, dim=-1))
        draw_probs = torch.stack(draw_probs)          # [n_draws, n_test, n_classes]
        mean_probs = draw_probs.mean(0)                 # posterior-averaged

    predictive_probs[label] = {"map": map_probs, "posterior_draws": draw_probs, "posterior_mean": mean_probs}

    map_metrics = classification_metrics(y_test, map_probs)
    post_metrics = classification_metrics(y_test, mean_probs)
    # per-test-point std of P(true class) across draws -- direct uncertainty readout
    p_true_per_draw = draw_probs.gather(-1, y_test.long().view(1, -1, 1).expand(draw_probs.shape[0], -1, 1)).squeeze(-1)
    disagreement = p_true_per_draw.std(dim=0).mean().item()

    perf_rows.append({
        "run": label,
        "map accuracy": map_metrics["accuracy"], "map log_lik": map_metrics["log_lik"], "map entropy": map_metrics["entropy"],
        "posterior accuracy": post_metrics["accuracy"], "posterior log_lik": post_metrics["log_lik"], "posterior entropy": post_metrics["entropy"],
        "ckpt test_accuracy": ckpt.get("test_accuracy"),
        "mean draw-to-draw P(true class) std": disagreement,
    })

perf_df = pd.DataFrame(perf_rows).set_index("run")
display(perf_df.style.format("{:.4f}"))

print(
    "Read: 'posterior entropy' vs 'map entropy' -- if the posterior-averaged prediction "
    "is meaningfully MORE uncertain (higher entropy) than the single MAP point at the "
    "same accuracy, the sampler is adding real epistemic-uncertainty signal, not just "
    "noise around the same point estimate. 'mean draw-to-draw P(true class) std' is the "
    "most direct 'do draws actually disagree with each other' number -- near 0 means the "
    "posterior draws are behaving like near-duplicates of x_ref on the test set, "
    "consistent with the small 'bounced' fraction found in sections 5/7 if it's small."
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for label, probs in predictive_probs.items():
    post_conf = probs["posterior_mean"].max(-1).values.numpy()
    map_conf = probs["map"].max(-1).values.numpy()
    axes[0].hist(post_conf, bins=30, alpha=0.5, label=f"{label} (posterior)", density=True)
    axes[0].hist(map_conf, bins=30, alpha=0.5, label=f"{label} (MAP)", density=True, histtype="step", lw=1.5)
axes[0].set_xlabel("max predicted class probability")
axes[0].set_title("Confidence distribution: posterior-avg vs. MAP")
axes[0].legend(fontsize=7)

for label, probs in predictive_probs.items():
    correct = (probs["posterior_mean"].argmax(-1) == y_test).numpy()
    conf = probs["posterior_mean"].max(-1).values.numpy()
    axes[1].scatter(conf[correct], np.zeros(correct.sum()) + 0.05, s=6, alpha=0.4, color="#55a868", label="correct" if label == list(predictive_probs)[0] else None)
    axes[1].scatter(conf[~correct], np.zeros((~correct).sum()) - 0.05, s=6, alpha=0.6, color="#c44e52", label="wrong" if label == list(predictive_probs)[0] else None)
axes[1].set_ylim(-1, 1)
axes[1].set_yticks([])
axes[1].set_xlabel("posterior-averaged max class probability")
axes[1].set_title("Confidence, correct vs. wrong predictions")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

print(
    "Read: a well-behaved posterior should push confidence DOWN specifically on the "
    "'wrong' points relative to 'correct' ones (red cluster left of green in the right "
    "panel) -- if wrong predictions are just as confident as correct ones, the posterior "
    "isn't buying calibration, whatever the accuracy number says."
)